# OpenEar Business Agent (Maguy)

## 0. Setup
Install libraries, load the API key from `.env`, and read the business files.

In [1]:
%pip install -q google-genai python-dotenv pypdf gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pypdf import PdfReader

load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")
assert api_key, "GEMINI_API_KEY not found. Check your .env file."

client = genai.Client(api_key=api_key)
MODEL = "gemini-3.5-flash"  

# Load the business knowledge the agent is allowed to use
with open("me/business_summary.txt", encoding="utf-8") as f:
    summary = f.read()

reader = PdfReader("me/about_business.pdf")
profile = "\n".join(page.extract_text() or "" for page in reader.pages)

print(f"Summary: {len(summary)} characters")
print(f"Profile: {len(profile)} characters, {len(reader.pages)} pages")

Summary: 2174 characters
Profile: 4688 characters, 3 pages


In [3]:
# Resilient Gemini call: retry, then fall back to another model on overload (503) or rate limit (429)
import time
from google.genai import errors

FALLBACK_MODELS = [MODEL, "gemini-3.5-flash-lite", "gemini-3.6-flash", "gemini-3.8-flash"]

def generate(contents, config=None, retries=2):
    last_error = None
    for model in dict.fromkeys(FALLBACK_MODELS):  # removes duplicates, keeps order
        for attempt in range(retries):
            try:
                return client.models.generate_content(model=model, contents=contents, config=config)
            except (errors.ServerError, errors.ClientError) as e:
                if getattr(e, "code", None) not in (429, 500, 503):
                    raise  # real errors (bad key, bad request) still surface
                last_error = e
                print(f"[{model} busy ({e.code}), retrying...]")
                time.sleep(2 * (attempt + 1))
    raise last_error

In [4]:
# Quick check that the key and model work (goes through the fallback helper)
response = generate("Reply with exactly: Maguy is online")
print(response.text)

[gemini-3.5-flash busy (503), retrying...]
[gemini-3.5-flash busy (503), retrying...]
Maguy is online


## 1. Persona
Who the agent is. Shapes tone and style.

In [5]:
# === PERSONA ===
PERSONA = """
You are Maguy, the sales assistant of OpenEar, a Lebanese online store for open-ear sports headphones.
You help athletes find the right headphones for their sport and guide them toward placing an order.
You are concise, friendly and sporty: short answers, no long paragraphs.
You match the customer's tone and language, replying in English, Arabic or Arabizi depending on how they write.
"""

## 2. Instructions
Operational guardrails the agent must follow.

In [6]:
# === INSTRUCTIONS ===
INSTRUCTIONS = """
1. Only use facts from the OpenEar business summary and business profile provided below.
2. Never invent prices, discounts, specs, stock, delivery times or policies.
3. Only answer questions about OpenEar's products, choosing headphones for a sport, orders, delivery, payment,
   warranty, returns and contact details. Politely decline anything else and steer back to OpenEar.
4. If a question is about OpenEar but the answer is not in the files, call record_feedback with the question,
   then tell the customer the team will follow up and share support@openear.store or @openear.lb on Instagram.
5. Before recommending a model, ask which sport the customer trains for if they haven't said.
   Then recommend: Stride for casual running, Pace Pro for serious running, trail and cycling,
   Swim for swimming and triathlon, Ultra for long-distance and ultra events.
6. When a customer wants to buy, order or reserve, or asks about availability, ask for their name and email,
   then call record_customer_interest with a short note of what they want.
7. If a customer showed buying interest but hasn't given an email, ask for it once before the conversation ends.
   Do not insist more than once.
8. Never ask for card numbers or payment details in the chat.
"""

## 3. Router
Before Maguy answers, a lightweight Gemini call classifies the message into one of three intents:
- **general**: questions about products, delivery, payment, warranty, returns
- **lead**: wants to buy, order or reserve, asks about availability, or is giving their name/email
- **feedback**: complaints, suggestions, or questions the business files cannot answer

The last two turns of the chat are included, so a reply like "Tatiana, t@mail.com" is recognised as a lead.

In [ ]:
INTENTS = ["general", "lead", "feedback"]

ROUTER_PROMPT = """You are a message router for OpenEar, a store selling open-ear sports headphones.
Classify the customer's LATEST message into exactly one intent:
- general: questions about products, specs, prices, delivery, payment, warranty, returns, or choosing a model
- lead: wants to buy, order or reserve, asks about availability, or is giving their name or email
- feedback: a complaint, a suggestion, or a question the store's information cannot answer
  (for example international shipping, discounts, bulk orders, or anything not covered)

Store information:
{knowledge}

Recent conversation:
{recent}

Latest message: {message}

Reply with one word only: general, lead or feedback."""


def route(message, history=None):
    """Classify a message into general / lead / feedback."""
    recent = ""
    for turn in (history or [])[-2:]:
        recent += f"{turn['role']}: {turn['content']}\n"
    prompt = ROUTER_PROMPT.format(knowledge=summary, recent=recent or "(none)", message=message)
    response = generate(prompt, config=types.GenerateContentConfig(temperature=0))
    label = (response.text or "").strip().lower()
    for intent in INTENTS:
        if intent in label:
            return intent
    return "general"  # safe default if the model replies with something unexpected

In [8]:
# Show the router working on sample messages
samples = [
    "What's the battery life of the Pace Pro?",
    "Do you deliver to Tripoli?",
    "I want to order the OpenEar Swim",
    "Tatiana, tatiana@mail.com",
    "Do you ship to Dubai?",
    "Do you have a student discount?",
    "My Stride stopped charging after a week",
]
for msg in samples:
    print(f"{route(msg):9} <- {msg}")

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## 4. Tools
Maguy has two tools. Each one prints a log line and appends a JSON record to a file in `logs/`, so the owner can review leads and unanswered questions later.

### Tool schema (MCP-style)

| Tool | Description | Parameters | Output |
|---|---|---|---|
| `record_customer_interest` | Saves a lead when a customer wants to buy, order or reserve and has shared their contact details. | `email` (string, required): customer email<br>`name` (string, required): customer name<br>`message` (string, required): what they want, e.g. "Pace Pro, black, marathon training" | `{"status": "recorded", "type": "lead"}` and a line in `logs/leads.jsonl` |
| `record_feedback` | Logs a question Maguy could not answer from the business files, or a complaint or suggestion. | `question` (string, required): the customer's question or feedback, in their words | `{"status": "recorded", "type": "feedback"}` and a line in `logs/feedback.jsonl` |

In [7]:
import json
from datetime import datetime

os.makedirs("logs", exist_ok=True)

def _append_log(filename, record):
    record["timestamp"] = datetime.now().isoformat(timespec="seconds")
    with open(os.path.join("logs", filename), "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def record_customer_interest(email, name, message):
    """Log a lead: a customer who wants to buy and shared their contact details."""
    print(f"[LEAD] {name} <{email}>: {message}")
    _append_log("leads.jsonl", {"name": name, "email": email, "message": message})
    return {"status": "recorded", "type": "lead"}


def record_feedback(question):
    """Log a question Maguy could not answer, or a complaint or suggestion."""
    print(f"[FEEDBACK] {question}")
    _append_log("feedback.jsonl", {"question": question})
    return {"status": "recorded", "type": "feedback"}


# JSON schemas sent to Gemini so the model knows when and how to call each tool
TOOL_SCHEMAS = [
    {
        "name": "record_customer_interest",
        "description": "Save a lead when a customer wants to buy, order or reserve an OpenEar product "
                       "and has given their name and email.",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "description": "The customer's email address"},
                "name": {"type": "string", "description": "The customer's name"},
                "message": {"type": "string", "description": "What the customer wants, e.g. product, colour, sport"},
            },
            "required": ["email", "name", "message"],
        },
    },
    {
        "name": "record_feedback",
        "description": "Log a customer question that the OpenEar business files cannot answer, "
                       "or a complaint or suggestion, so the team can follow up.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {"type": "string", "description": "The customer's question or feedback, in their words"},
            },
            "required": ["question"],
        },
    },
]

TOOLS = types.Tool(function_declarations=TOOL_SCHEMAS)
TOOL_FUNCTIONS = {
    "record_customer_interest": record_customer_interest,
    "record_feedback": record_feedback,
}
print("Tools ready:", list(TOOL_FUNCTIONS))

Tools ready: ['record_customer_interest', 'record_feedback']


In [8]:
# Call each tool once by hand to check the logging works
print(record_customer_interest("test@example.com", "Test Customer", "Pace Pro, black, marathon training"))
print(record_feedback("Do you ship to Dubai?"))

[LEAD] Test Customer <test@example.com>: Pace Pro, black, marathon training
{'status': 'recorded', 'type': 'lead'}
[FEEDBACK] Do you ship to Dubai?
{'status': 'recorded', 'type': 'feedback'}


## 5. System prompt and chat loop
The system prompt combines the persona, the instructions and the business files.

For every customer message, the agent:
1. **Routes** the message (general / lead / feedback) and passes the intent to the model as a hint.
2. **Calls Gemini** with the system prompt, the conversation history and the tool schemas.
3. **Runs any tool calls** the model makes (`record_customer_interest`, `record_feedback`), sends the results back, and lets the model write its final reply.

In [ ]:
SYSTEM_PROMPT = f"""{PERSONA}

## Instructions
{INSTRUCTIONS}

## Business summary
{summary}

## Business profile
{profile}
"""

INTENT_HINTS = {
    "general": "Answer from the business files.",
    "lead": "The customer shows buying interest. If you have their name and email, call record_customer_interest. "
            "Otherwise ask for them.",
    "feedback": "This is feedback or a question the files may not answer. If the files don't answer it, "
                "call record_feedback, then tell the customer the team will follow up.",
}


def to_contents(history):
    """Convert [{'role': 'user'|'assistant', 'content': str}] into Gemini contents."""
    return [
        types.Content(role="model" if t["role"] == "assistant" else "user",
                      parts=[types.Part.from_text(text=t["content"])])
        for t in history
    ]


def chat(message, history, verbose=True):
    """One customer turn: route, call Gemini, run tools, return Maguy's reply."""
    intent = route(message, history)
    if verbose:
        print(f"[ROUTER] {intent}")

    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT + f"\n## Router intent for this message: {intent}\n{INTENT_HINTS[intent]}",
        tools=[TOOLS],
        temperature=0.4,
    )
    contents = to_contents(history) + [
        types.Content(role="user", parts=[types.Part.from_text(text=message)])
    ]

    for _ in range(5):  # at most 5 tool rounds per message
        response = generate(contents, config=config)
        calls = response.function_calls or []
        if not calls:
            return response.text
        contents.append(response.candidates[0].content)  # keep the model's tool-call turn
        results = []
        for call in calls:
            func = TOOL_FUNCTIONS.get(call.name)
            result = func(**dict(call.args)) if func else {"error": f"unknown tool {call.name}"}
            results.append(types.Part.from_function_response(name=call.name, response=result))
        contents.append(types.Content(role="user", parts=results))

    return "Sorry, something went wrong. Please contact support@openear.store."

In [ ]:
# Simulated conversation: recommendation, unanswered question, then a lead
history = []
for msg in [
    "Hi! Which headphones should I get?",
    "I mostly swim",
    "Do you ship to Dubai?",
    "OK I'll take the Swim. Tatiana, tatiana@example.com",
]:
    reply = chat(msg, history)
    print(f"Customer: {msg}\nMaguy: {reply}\n")
    history += [{"role": "user", "content": msg}, {"role": "assistant", "content": reply}]